# SHAP-Guided Hypothesis Testing

기존 `Hypothesis_Testing.ipynb`를 뼈대로 두고,
`shap_pipeline.ipynb`에서 반복적으로 살아남은 Robust 피처를 앞단 근거로 붙인 발표용 노트북입니다.

이번 버전의 원칙은 두 가지입니다.
1. SHAP에서 반복적으로 살아남은 피처만 가설의 주연·조연으로 올립니다.
2. 첫 결과를 그대로 믿지 않고, 각 검정마다 허점과 민감도를 먼저 점검합니다.

## 이번 노트북의 3개 가설

### H1. 관로/팁 막힘
- 메인 변수: `zone1_resistance`
- 보조 변수: `filter_delta_p_kpa`, `flow_drop_rate`, `zone1_flow_l_min`
- 검정 기법: **Granger Causality Test**

### H2. 채널링/미세 누수
- 메인 변수: `drain_ec_ds_m`
- 보조 변수: `zone1_substrate_ec_ds_m`, `zone1_substrate_moisture_pct`
- 검정 기법: **Event-window CCF**
- 주의: CCF를 그대로 믿기 전에 onset 규칙 민감도와 이벤트 구조를 먼저 비판적으로 점검합니다.

### H3. 펌프 노후화/기계 열화
- 메인 변수: `wire_to_water_efficiency`
- 보조 변수: `motor_current_a`, `bearing_vibration_rms_mm_s`, `motor_temperature_c`
- 검정 기법: **Welch T-test + Mann-Whitney U**


## 0. 공통 함수 준비

기존 가설검정 노트북의 계산 함수를 그대로 재사용하되,
아래 셀 끝에서 SHAP/Robust 피처 앵커와 H2 CCF용 보조 함수를 추가로 덧붙입니다.


In [ ]:
from IPython.display import display, Markdown, Image

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager
from scipy.optimize import minimize_scalar
from scipy.stats import f as f_dist
from scipy.stats import mannwhitneyu, norm, ttest_ind


ALPHA = 0.05
GRANGER_MAX_LAG = 6
EPS = 1e-6

PUMP_ON_RPM_THRESHOLD = 100
PRE_OFF_MINUTES = 60
POST_ON_MINUTES = 60
MAX_RESPONSE_LAG_MINUTES = 60
CONSECUTIVE_HITS = 2
WEEK_DAYS = 7

CANDIDATE_PATHS = [
    Path("human_A/src/selected_smartfarm.csv"),
    Path("src/selected_smartfarm.csv"),
    Path("selected_smartfarm.csv"),
]

OUT_DIR = Path("human_A") / "hypothesis_testing_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.unicode_minus"] = False

MALGUN_PATH = Path(r"C:\Windows\Fonts\malgun.ttf")
if MALGUN_PATH.exists():
    font_manager.fontManager.addfont(str(MALGUN_PATH))
    KOREAN_FONT = font_manager.FontProperties(fname=str(MALGUN_PATH)).get_name()
else:
    KOREAN_FONT = "Malgun Gothic"
plt.rcParams["font.family"] = KOREAN_FONT
plt.rcParams["font.sans-serif"] = [KOREAN_FONT, "DejaVu Sans"]


def dataframe_to_simple_markdown(df):
    if df.empty:
        return "_empty_"
    headers = [str(col) for col in df.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for _, row in df.iterrows():
        values = ["" if pd.isna(v) else str(v) for v in row]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


def resolve_csv_path(paths):
    """실제로 존재하는 첫 번째 CSV 경로를 찾습니다."""
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(paths)


def load_raw_data(path):
    """원시 CSV를 읽고 시간순으로 정렬합니다."""
    df = pd.read_csv(path, parse_dates=["timestamp"])
    return df.sort_values("timestamp").set_index("timestamp")


def aggregate_10m_tumbling(df):
    """1분 데이터를 10분 단위 평균으로 묶습니다."""
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    return df[numeric_cols].resample("10min").mean().dropna()


def create_features_after_agg(df):
    """10분 집계 후 H1/H3용 파생변수를 다시 계산합니다."""
    feat = df.copy()
    dt_s = feat.index.to_series().diff().dt.total_seconds().fillna(600)

    feat["pump_on_proxy"] = (feat["pump_rpm"] > PUMP_ON_RPM_THRESHOLD).astype(int)
    feat["differential_pressure_kpa"] = feat["discharge_pressure_kpa"] - feat["suction_pressure_kpa"]
    feat["filter_delta_p_kpa"] = feat["filter_pressure_in_kpa"] - feat["filter_pressure_out_kpa"]
    feat["hydraulic_power_kw"] = (feat["flow_rate_l_min"] * feat["differential_pressure_kpa"]) / 60000
    feat["wire_to_water_efficiency"] = feat["hydraulic_power_kw"] / (feat["motor_power_kw"] + EPS)
    feat["flow_baseline_l_min"] = feat["flow_rate_l_min"].rolling(6, min_periods=1).mean().shift(1)
    feat["flow_drop_rate"] = (feat["flow_baseline_l_min"] - feat["flow_rate_l_min"]) / (feat["flow_baseline_l_min"] + EPS)
    feat["temp_slope_c_per_s"] = feat["motor_temperature_c"].diff() / dt_s
    rpm_mean = feat["pump_rpm"].rolling(3, min_periods=1).mean()
    feat["rpm_stability_index"] = np.abs(feat["pump_rpm"] - rpm_mean) / (rpm_mean + EPS)
    feat["zone1_resistance"] = feat["zone1_pressure_kpa"] / (feat["zone1_flow_l_min"] + EPS)
    feat["zone1_moisture_response_pct"] = feat["zone1_substrate_moisture_pct"].diff()
    return feat.replace([np.inf, -np.inf], np.nan)


def granger_f_test(data, x_col, y_col, max_lag=6, difference=True):
    """간단한 직접 계산 Granger F-test입니다."""
    z = data[[x_col, y_col]].dropna().copy()
    if difference:
        z = z.diff().dropna()
    rows = []
    for lag in range(1, max_lag + 1):
        w = z.copy()
        for i in range(1, lag + 1):
            w[f"{y_col}_lag_{i}"] = w[y_col].shift(i)
            w[f"{x_col}_lag_{i}"] = w[x_col].shift(i)
        w = w.dropna()
        if len(w) < 80:
            continue
        y = w[y_col].to_numpy(dtype=float)
        x_r = np.column_stack([np.ones(len(w))] + [w[f"{y_col}_lag_{i}"].to_numpy(dtype=float) for i in range(1, lag + 1)])
        x_u = np.column_stack(
            [np.ones(len(w))]
            + [w[f"{y_col}_lag_{i}"].to_numpy(dtype=float) for i in range(1, lag + 1)]
            + [w[f"{x_col}_lag_{i}"].to_numpy(dtype=float) for i in range(1, lag + 1)]
        )
        beta_r, *_ = np.linalg.lstsq(x_r, y, rcond=None)
        beta_u, *_ = np.linalg.lstsq(x_u, y, rcond=None)
        rss_r = np.sum((y - x_r @ beta_r) ** 2)
        rss_u = np.sum((y - x_u @ beta_u) ** 2)
        df1 = lag
        df2 = len(w) - x_u.shape[1]
        if df2 <= 0 or rss_u <= 0:
            continue
        f_stat = ((rss_r - rss_u) / df1) / (rss_u / df2)
        p_value = f_dist.sf(f_stat, df1, df2)
        rows.append(
            {
                "lag_10m": lag,
                "lag_minutes": lag * 10,
                "n_obs": len(w),
                "f_stat": float(f_stat),
                "p_value": float(p_value),
                "significant": bool(p_value < ALPHA),
            }
        )
    return pd.DataFrame(rows)


def get_best_granger_result(result):
    if result.empty:
        return {"lag_10m": None, "lag_minutes": None, "p_value": None, "significant": False}
    best = result.sort_values("p_value").iloc[0]
    return {
        "lag_10m": int(best["lag_10m"]),
        "lag_minutes": int(best["lag_minutes"]),
        "p_value": float(best["p_value"]),
        "significant": bool(best["significant"]),
    }


def get_on_mask(df):
    """H2에서 공급 시작 블록을 정의합니다."""
    return df["pump_rpm"] > PUMP_ON_RPM_THRESHOLD


def exact_hold_run_quantiles(series):
    """센서가 같은 값을 얼마나 오래 유지하는지 봅니다."""
    changed = series.ne(series.shift())
    run_id = changed.cumsum()
    run_sizes = series.groupby(run_id).size()
    return run_sizes.quantile([0.5, 0.9, 0.95, 0.99, 0.999])


def compute_mad_threshold(abs_diff_signal, inactive_mask):
    """꺼진 시간대 노이즈로 robust threshold를 계산합니다."""
    baseline = abs_diff_signal.loc[inactive_mask].dropna()
    median = baseline.median()
    mad = (baseline - median).abs().median()
    sigma = baseline.std(ddof=1)
    threshold = 3 * 1.4826 * mad
    return {
        "baseline_n": int(len(baseline)),
        "median": float(median),
        "mad": float(mad),
        "sigma": float(sigma),
        "threshold_mad": float(threshold),
        "threshold_sigma": float(3 * sigma),
    }


def find_isolated_event_starts(on_mask):
    """직전 60분 off, 직후 60분 on 조건을 만족하는 이벤트 시작만 남깁니다."""
    starts = on_mask.index[on_mask & ~on_mask.shift(fill_value=False)]
    kept = []
    first_ts = on_mask.index[0]
    last_ts = on_mask.index[-1]
    for ts in starts:
        pre_start = ts - pd.Timedelta(minutes=PRE_OFF_MINUTES)
        post_end = ts + pd.Timedelta(minutes=POST_ON_MINUTES - 1)
        if pre_start < first_ts or post_end > last_ts:
            continue
        pre_ok = (~on_mask.loc[pre_start : ts - pd.Timedelta(minutes=1)]).all()
        post_ok = on_mask.loc[ts:post_end].all()
        if pre_ok and post_ok:
            kept.append(ts)
    return pd.DatetimeIndex(kept)


def first_consecutive_hit(signal, start_ts, threshold, max_lag_minutes=60, consecutive_hits=2):
    """2분 연속 threshold 초과가 처음 나타나는 onset 시점을 찾습니다."""
    window = signal.loc[start_ts : start_ts + pd.Timedelta(minutes=max_lag_minutes)].copy()
    above = (window > threshold).astype(int)
    rolling_hits = above.rolling(consecutive_hits, min_periods=consecutive_hits).sum()
    hit_idx = rolling_hits[rolling_hits >= consecutive_hits].index
    if len(hit_idx) == 0:
        return None
    onset_end = hit_idx[0]
    return onset_end - pd.Timedelta(minutes=consecutive_hits - 1)


def build_event_lag_table(df_raw, abs_diff_signal, threshold):
    """각 이벤트의 lag와 censoring 여부를 표로 만듭니다."""
    on_mask = get_on_mask(df_raw)
    starts = find_isolated_event_starts(on_mask)
    rows = []
    for event_idx, ts in enumerate(starts, start=1):
        onset_ts = first_consecutive_hit(
            signal=abs_diff_signal,
            start_ts=ts,
            threshold=threshold,
            max_lag_minutes=MAX_RESPONSE_LAG_MINUTES,
            consecutive_hits=CONSECUTIVE_HITS,
        )
        observed = onset_ts is not None
        lag_minutes = (onset_ts - ts).total_seconds() / 60 if observed else MAX_RESPONSE_LAG_MINUTES
        rows.append(
            {
                "event_index": event_idx,
                "event_index_10": (event_idx - 1) / 10.0,
                "event_start": ts,
                "lag_minutes": float(lag_minutes),
                "event_observed": int(observed),
                "event_date": ts.date(),
            }
        )
    return pd.DataFrame(rows)


def kaplan_meier_table(durations, events):
    """간단한 KM 생존함수 표입니다."""
    data = pd.DataFrame({"duration": durations, "event": events}).sort_values("duration")
    unique_times = np.sort(data.loc[data["event"] == 1, "duration"].unique())
    at_risk = len(data)
    survival = 1.0
    rows = [{"time": 0.0, "survival": 1.0, "n_risk": at_risk, "n_event": 0}]
    for t in unique_times:
        d_t = int(((data["duration"] == t) & (data["event"] == 1)).sum())
        c_t = int(((data["duration"] == t) & (data["event"] == 0)).sum())
        if at_risk > 0:
            survival *= 1 - d_t / at_risk
        rows.append({"time": float(t), "survival": float(survival), "n_risk": int(at_risk), "n_event": d_t})
        at_risk -= d_t + c_t
    return pd.DataFrame(rows)


def cox_ph_single_covariate_breslow(durations, events, covariate):
    """외부 패키지 없이 1개 공변량 Cox PH를 계산합니다."""
    durations = np.asarray(durations, dtype=float)
    events = np.asarray(events, dtype=int)
    covariate = np.asarray(covariate, dtype=float)
    if events.sum() == 0:
        return {"beta": np.nan, "se": np.nan, "z": np.nan, "p_value": np.nan, "hazard_ratio": np.nan, "ci_low": np.nan, "ci_high": np.nan}
    event_times = np.sort(np.unique(durations[events == 1]))

    def neg_log_partial_likelihood(beta):
        eta = beta * covariate
        risk = np.exp(np.clip(eta, -50, 50))
        total = 0.0
        for t in event_times:
            event_mask = (durations == t) & (events == 1)
            d_t = event_mask.sum()
            risk_mask = durations >= t
            total += eta[event_mask].sum()
            total -= d_t * np.log(risk[risk_mask].sum() + EPS)
        return -total

    def observed_information(beta):
        eta = beta * covariate
        risk = np.exp(np.clip(eta, -50, 50))
        info = 0.0
        for t in event_times:
            event_mask = (durations == t) & (events == 1)
            d_t = event_mask.sum()
            risk_mask = durations >= t
            risk_sum = risk[risk_mask].sum() + EPS
            mean_x = (risk[risk_mask] * covariate[risk_mask]).sum() / risk_sum
            mean_x2 = (risk[risk_mask] * (covariate[risk_mask] ** 2)).sum() / risk_sum
            info += d_t * (mean_x2 - mean_x**2)
        return info

    opt = minimize_scalar(neg_log_partial_likelihood, bracket=(-2, 0, 2), method="brent")
    beta = float(opt.x)
    info = observed_information(beta)
    if info <= 0:
        se = z = p_value = ci_low = ci_high = np.nan
    else:
        se = float(np.sqrt(1 / info))
        z = float(beta / se)
        p_value = float(2 * norm.sf(abs(z)))
        ci_low = float(np.exp(beta - 1.96 * se))
        ci_high = float(np.exp(beta + 1.96 * se))
    return {
        "beta": beta,
        "se": se,
        "z": z,
        "p_value": p_value,
        "hazard_ratio": float(np.exp(beta)),
        "ci_low": ci_low,
        "ci_high": ci_high,
    }


def auxiliary_event_ccf(df_raw, event_starts, abs_diff_signal, max_lag_minutes=60):
    """H2 보조용 sparse-window CCF입니다."""
    rows = []
    for lag in range(0, max_lag_minutes + 1):
        corrs = []
        for ts in event_starts:
            x = df_raw.loc[ts : ts + pd.Timedelta(minutes=max_lag_minutes), "mix_flow_l_min"].to_numpy(dtype=float)
            y = abs_diff_signal.loc[ts : ts + pd.Timedelta(minutes=max_lag_minutes)].to_numpy(dtype=float)
            if len(x) < max_lag_minutes + 1 or len(y) < max_lag_minutes + 1:
                continue
            if lag == 0:
                x_now, y_now = x, y
            else:
                x_now, y_now = x[:-lag], y[lag:]
            if x_now.std() < 1e-9 or y_now.std() < 1e-9:
                continue
            x_now = (x_now - x_now.mean()) / (x_now.std() + 1e-9)
            y_now = (y_now - y_now.mean()) / (y_now.std() + 1e-9)
            corrs.append(float(np.corrcoef(x_now, y_now)[0, 1]))
        rows.append({"lag_minutes": lag, "mean_corr": float(np.mean(corrs)) if corrs else np.nan, "n_windows": int(len(corrs))})
    curve = pd.DataFrame(rows)
    valid = curve.dropna(subset=["mean_corr"])
    if valid.empty:
        peak = {"lag_minutes": None, "mean_corr": None}
    else:
        peak_row = valid.loc[valid["mean_corr"].abs().idxmax()]
        peak = {"lag_minutes": int(peak_row["lag_minutes"]), "mean_corr": float(peak_row["mean_corr"])}
    return curve, peak


def build_h3_groups(df_feat):
    """H3 비교용 그룹을 만듭니다."""
    active = df_feat.loc[df_feat["pump_on_proxy"] == 1].copy()
    active["date"] = active.index.date
    daily_mean = active.groupby("date")[["wire_to_water_efficiency", "bearing_vibration_rms_mm_s", "motor_temperature_c"]].mean()
    daily_mean["day_idx"] = np.arange(len(daily_mean))
    mode = "time_proxy_first7_vs_last7"
    group_a = daily_mean.head(WEEK_DAYS).copy()
    group_b = daily_mean.tail(WEEK_DAYS).copy()
    if "cleaning_event_flag" in df_feat.columns:
        cleaning_days = df_feat.loc[df_feat["cleaning_event_flag"] > 0].index.normalize().unique()
        if len(cleaning_days) > 0:
            clean_day = pd.Timestamp(cleaning_days[0]).date()
            if clean_day in daily_mean.index:
                pos = list(daily_mean.index).index(clean_day)
                if pos >= WEEK_DAYS and pos + WEEK_DAYS < len(daily_mean):
                    mode = "true_cleaning_prepost"
                    group_a = daily_mean.iloc[pos - WEEK_DAYS : pos].copy()
                    group_b = daily_mean.iloc[pos + 1 : pos + 1 + WEEK_DAYS].copy()
    return daily_mean, group_a, group_b, mode


def run_phase0_checks(df_raw):
    """H2 사전 적합성 점검을 실행합니다."""
    on_mask = get_on_mask(df_raw)
    inactive_mask = ~on_mask
    drain_abs_diff = df_raw["drain_ec_ds_m"].diff().abs()
    cadence_quantiles = exact_hold_run_quantiles(df_raw["drain_ec_ds_m"])
    threshold_info = compute_mad_threshold(drain_abs_diff, inactive_mask)
    isolated_starts = find_isolated_event_starts(on_mask)
    lag_table = build_event_lag_table(df_raw, drain_abs_diff, threshold_info["threshold_mad"])
    zone_rows = []
    zone_signals = [
        "zone1_substrate_moisture_pct",
        "zone2_substrate_moisture_pct",
        "zone3_substrate_moisture_pct",
        "zone1_substrate_ec_ds_m",
        "zone2_substrate_ec_ds_m",
        "zone3_substrate_ec_ds_m",
    ]
    for col in zone_signals:
        zone_abs_diff = df_raw[col].diff().abs()
        zone_thr = compute_mad_threshold(zone_abs_diff, inactive_mask)["threshold_mad"]
        zone_lag_table = build_event_lag_table(df_raw, zone_abs_diff, zone_thr)
        failure_rate = 1 - zone_lag_table["event_observed"].mean()
        zone_rows.append({"signal": col, "threshold": float(zone_thr), "events": int(len(zone_lag_table)), "failure_rate": float(failure_rate), "pass_gate": bool(failure_rate <= 0.30)})

    drain_failure_rate = float(1 - lag_table["event_observed"].mean())
    h2_main_pass = bool(drain_failure_rate <= 0.30)

    phase0_summary = pd.DataFrame(
        [
            {"metric": "drain_ec_effective_cadence_median_hold_min", "value": float(cadence_quantiles.loc[0.5])},
            {"metric": "drain_ec_effective_cadence_p95_hold_min", "value": float(cadence_quantiles.loc[0.95])},
            {"metric": "isolated_event_count", "value": int(len(isolated_starts))},
            {"metric": "isolated_event_ratio", "value": float(len(isolated_starts) / max(int((on_mask & ~on_mask.shift(fill_value=False)).sum()), 1))},
            {"metric": "drain_threshold_mad", "value": threshold_info["threshold_mad"]},
            {"metric": "drain_threshold_sigma", "value": threshold_info["threshold_sigma"]},
            {"metric": "drain_failure_rate", "value": drain_failure_rate},
            {"metric": "h2_main_gate_pass", "value": h2_main_pass},
        ]
    )

    phase0_summary.to_csv(OUT_DIR / "phase0_summary.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame([threshold_info]).to_csv(OUT_DIR / "phase0_thresholds.csv", index=False, encoding="utf-8-sig")
    lag_table.to_csv(OUT_DIR / "h2_event_lag_table.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(zone_rows).to_csv(OUT_DIR / "h2_zone_signal_availability.csv", index=False, encoding="utf-8-sig")
    return {
        "phase0_summary": phase0_summary,
        "threshold_info": threshold_info,
        "isolated_starts": isolated_starts,
        "lag_table": lag_table,
        "zone_summary": pd.DataFrame(zone_rows),
        "drain_abs_diff": drain_abs_diff,
        "h2_main_pass": h2_main_pass,
        "drain_failure_rate": drain_failure_rate,
    }


def run_h1(df_feat):
    """H1 원안과 대체축을 계산합니다."""
    df_active = df_feat.loc[df_feat["pump_on_proxy"] == 1].copy()
    df_active_h1 = df_active.loc[df_active["zone1_flow_l_min"] > 1.0].copy()
    h1_original = granger_f_test(df_active_h1, "zone1_resistance", "zone1_moisture_response_pct", max_lag=GRANGER_MAX_LAG, difference=True)
    h1_alt = granger_f_test(df_active, "filter_delta_p_kpa", "flow_drop_rate", max_lag=GRANGER_MAX_LAG, difference=True)
    h1_summary = pd.DataFrame(
        [
            {"hypothesis": "H1_original", "x": "zone1_resistance", "y": "zone1_moisture_response_pct", **get_best_granger_result(h1_original)},
            {"hypothesis": "H1_alternative", "x": "filter_delta_p_kpa", "y": "flow_drop_rate", **get_best_granger_result(h1_alt)},
        ]
    )
    h1_summary.to_csv(OUT_DIR / "h1_summary.csv", index=False, encoding="utf-8-sig")
    h1_original.to_csv(OUT_DIR / "h1_granger_original.csv", index=False, encoding="utf-8-sig")
    h1_alt.to_csv(OUT_DIR / "h1_granger_alternative.csv", index=False, encoding="utf-8-sig")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    for ax, result, title in zip(axes, [h1_original, h1_alt], ["H1 원안", "H1 대체축"]):
        if not result.empty:
            sns.lineplot(data=result, x="lag_minutes", y="p_value", marker="o", ax=ax)
            ax.axhline(ALPHA, color="red", linestyle="--", linewidth=1)
        ax.set_title(title)
        ax.set_xlabel("lag (minutes)")
        ax.set_ylabel("p-value")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "h1_granger.png", dpi=150, bbox_inches="tight")
    plt.close()
    return {"summary": h1_summary, "original": h1_original, "alternative": h1_alt}


def run_h2(df_raw, phase0_result):
    """H2 메인 분석(Cox/KM)과 보조 CCF를 계산합니다."""
    lag_table = phase0_result["lag_table"].copy()
    isolated_starts = phase0_result["isolated_starts"]
    drain_abs_diff = phase0_result["drain_abs_diff"]
    cox_result = cox_ph_single_covariate_breslow(
        durations=lag_table["lag_minutes"].to_numpy(),
        events=lag_table["event_observed"].to_numpy(),
        covariate=lag_table["event_index_10"].to_numpy(),
    )
    cox_summary = pd.DataFrame(
        [
            {
                "covariate": "event_index_per_10_events",
                "beta": cox_result["beta"],
                "se": cox_result["se"],
                "z": cox_result["z"],
                "p_value": cox_result["p_value"],
                "hazard_ratio": cox_result["hazard_ratio"],
                "ci_low": cox_result["ci_low"],
                "ci_high": cox_result["ci_high"],
            }
        ]
    )
    cox_summary.to_csv(OUT_DIR / "h2_cox_summary.csv", index=False, encoding="utf-8-sig")

    split_point = int(np.ceil(len(lag_table) / 2))
    early = lag_table.iloc[:split_point].copy()
    late = lag_table.iloc[split_point:].copy()
    km_early = kaplan_meier_table(early["lag_minutes"], early["event_observed"])
    km_late = kaplan_meier_table(late["lag_minutes"], late["event_observed"])
    km_early["group"] = "early_half"
    km_late["group"] = "late_half"
    km_all = pd.concat([km_early, km_late], ignore_index=True)
    km_all.to_csv(OUT_DIR / "h2_kaplan_meier.csv", index=False, encoding="utf-8-sig")
    plt.figure(figsize=(8, 4))
    for group_name, km_df in [("early_half", km_early), ("late_half", km_late)]:
        plt.step(km_df["time"], km_df["survival"], where="post", label=group_name)
    plt.xlabel("lag (minutes)")
    plt.ylabel("survival: 아직 반응 없음")
    plt.title("H2 Kaplan-Meier (전반기 vs 후반기)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / "h2_kaplan_meier.png", dpi=150, bbox_inches="tight")
    plt.close()

    ccf_curve, ccf_peak = auxiliary_event_ccf(df_raw, isolated_starts, drain_abs_diff, max_lag_minutes=MAX_RESPONSE_LAG_MINUTES)
    ccf_curve.to_csv(OUT_DIR / "h2_auxiliary_ccf.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame([ccf_peak]).to_csv(OUT_DIR / "h2_auxiliary_ccf_peak.csv", index=False, encoding="utf-8-sig")
    plt.figure(figsize=(8, 4))
    sns.lineplot(data=ccf_curve, x="lag_minutes", y="mean_corr", marker="o")
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.title("H2 보조 CCF: mix_flow vs |Δdrain_ec|")
    plt.xlabel("lag (minutes)")
    plt.ylabel("mean correlation")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "h2_auxiliary_ccf.png", dpi=150, bbox_inches="tight")
    plt.close()

    boundary_share = float(((lag_table["event_observed"] == 1) & (lag_table["lag_minutes"] == MAX_RESPONSE_LAG_MINUTES)).mean())
    pd.DataFrame([{"metric": "detected_at_right_boundary_share", "value": boundary_share}]).to_csv(
        OUT_DIR / "h2_boundary_check.csv", index=False, encoding="utf-8-sig"
    )
    return {"lag_table": lag_table, "cox_summary": cox_summary, "km_table": km_all, "ccf_peak": ccf_peak, "boundary_share": boundary_share}


def run_h3(df_feat):
    """H3 T-test / Mann-Whitney를 계산합니다."""
    daily_mean, group_a, group_b, mode = build_h3_groups(df_feat)
    trend_rows = []
    for col in ["wire_to_water_efficiency", "bearing_vibration_rms_mm_s", "motor_temperature_c"]:
        corr = np.corrcoef(daily_mean["day_idx"], daily_mean[col])[0, 1]
        trend_rows.append({"metric": col, "corr_with_time": float(corr)})
    trend_df = pd.DataFrame(trend_rows)
    welch = ttest_ind(group_a["wire_to_water_efficiency"], group_b["wire_to_water_efficiency"], equal_var=False)
    mw = mannwhitneyu(group_a["wire_to_water_efficiency"], group_b["wire_to_water_efficiency"], alternative="two-sided")
    h3_summary = pd.DataFrame(
        [
            {"mode": mode, "group": "group_a", "n_days": len(group_a), "mean_efficiency": group_a["wire_to_water_efficiency"].mean(), "mean_vibration": group_a["bearing_vibration_rms_mm_s"].mean(), "mean_motor_temp": group_a["motor_temperature_c"].mean()},
            {"mode": mode, "group": "group_b", "n_days": len(group_b), "mean_efficiency": group_b["wire_to_water_efficiency"].mean(), "mean_vibration": group_b["bearing_vibration_rms_mm_s"].mean(), "mean_motor_temp": group_b["motor_temperature_c"].mean()},
        ]
    )
    h3_tests = pd.DataFrame([{"mode": mode, "welch_t_p_value": float(welch.pvalue), "mann_whitney_p_value": float(mw.pvalue)}])
    trend_df.to_csv(OUT_DIR / "h3_trend_summary.csv", index=False, encoding="utf-8-sig")
    h3_summary.to_csv(OUT_DIR / "h3_group_summary.csv", index=False, encoding="utf-8-sig")
    h3_tests.to_csv(OUT_DIR / "h3_test_summary.csv", index=False, encoding="utf-8-sig")
    plot_df = pd.concat([group_a.assign(group="group_a"), group_b.assign(group="group_b")]).reset_index(names="date")
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=plot_df, x="group", y="wire_to_water_efficiency")
    plt.title("H3: wire_to_water_efficiency 비교")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "h3_ttest.png", dpi=150, bbox_inches="tight")
    plt.close()
    return {"trend_summary": trend_df, "group_summary": h3_summary, "test_summary": h3_tests, "mode": mode}


def write_markdown_report(csv_path, phase0_result, h1_result, h2_result, h3_result):
    """발표용 요약 리포트를 만듭니다."""
    h1_summary = h1_result["summary"]
    h2_cox = h2_result["cox_summary"].iloc[0]
    h3_test = h3_result["test_summary"].iloc[0]
    zone_summary = phase0_result["zone_summary"]
    lag_table = h2_result["lag_table"]
    h2_status = "통과" if phase0_result["h2_main_pass"] else "탐색적 분석으로 강등"
    lines = [
        "# Hypothesis Testing Summary",
        "",
        f"- 입력 파일: `{csv_path}`",
        f"- 유의수준: `{ALPHA}`",
        "",
        "## Phase 0",
        f"- drain_ec median hold 길이: `{phase0_result['phase0_summary'].loc[phase0_result['phase0_summary']['metric']=='drain_ec_effective_cadence_median_hold_min','value'].iloc[0]:.1f}분`",
        f"- isolated event 수: `{len(phase0_result['isolated_starts'])}`",
        f"- MAD threshold: `{phase0_result['threshold_info']['threshold_mad']:.6f}`",
        f"- drain lag 검출 실패율: `{phase0_result['drain_failure_rate']:.3f}`",
        f"- H2 메인 게이트 판정: `{h2_status}`",
        "",
        "## H1 Granger",
        dataframe_to_simple_markdown(h1_summary),
        "",
        "## H2 Event-based lag",
        f"- Cox hazard ratio (10 event 증가당): `{h2_cox['hazard_ratio']:.4f}`",
        f"- Cox p-value: `{h2_cox['p_value']:.6g}`",
        f"- 우측 경계(60분) 검출 비율: `{h2_result['boundary_share']:.3f}`",
        f"- 보조 CCF peak lag: `{h2_result['ccf_peak']['lag_minutes']}`분",
        f"- 보조 CCF peak corr: `{h2_result['ccf_peak']['mean_corr']}`",
        "",
        "### Zone reproducibility",
        dataframe_to_simple_markdown(zone_summary),
        "",
        "## H3 T-test / Mann-Whitney",
        f"- 비교 모드: `{h3_result['mode']}`",
        f"- Welch t-test p-value: `{h3_test['welch_t_p_value']:.6g}`",
        f"- Mann-Whitney p-value: `{h3_test['mann_whitney_p_value']:.6g}`",
        "",
        "## 해석 제한",
        "- H2는 clogging의 직접 인과 증명이 아니라 repeated-event response lag 패턴 분석입니다.",
        "- 현재 고정 onset 규칙(2분 연속 초과)을 적용하면 H2 메인 실패율이 30%를 넘어, 확증적 검정보다 탐색적 분석으로 보는 편이 안전합니다.",
        "- 이 데이터에서는 운영 일수와 이벤트 수가 사실상 같은 축이어서 달력 시간 효과와 반복 운영 효과를 분리할 수 없습니다.",
        "- H3는 cleaning_event_flag가 없으면 세척 전후가 아니라 time-based proxy 비교로만 해석해야 합니다.",
    ]
    (OUT_DIR / "hypothesis_summary.md").write_text("\n".join(lines), encoding="utf-8-sig")


    csv_path = resolve_csv_path(CANDIDATE_PATHS)
    df_raw = load_raw_data(csv_path)
    df_10m = aggregate_10m_tumbling(df_raw)
    df_feat = create_features_after_agg(df_10m)
    phase0_result = run_phase0_checks(df_raw)
    h1_result = run_h1(df_feat)
    h2_result = run_h2(df_raw, phase0_result)
    h3_result = run_h3(df_feat)
    write_markdown_report(csv_path, phase0_result, h1_result, h2_result, h3_result)
    print("CSV_PATH:", csv_path)
    print("Raw shape:", df_raw.shape)
    print("10-minute aggregated shape:", df_10m.shape)
    print("Feature shape:", df_feat.shape)
    print("결과 저장 위치:", OUT_DIR)


BASE_NOTEBOOK_DIR = Path("human_A") if Path("human_A").exists() else Path(".")
OUT_DIR = BASE_NOTEBOOK_DIR / "shap_guided_hypothesis_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ROBUST_IMG_CANDIDATES = [
    Path("output_m") / "robust_per_target.png",
    Path("..") / "output_m" / "robust_per_target.png",
]
SHAP_IMG_CANDIDATES = [
    Path("output_m") / "shap_top10.png",
    Path("..") / "output_m" / "shap_top10.png",
]

ROBUST_FEATURE_TABLE = [
    {"feature": "zone1_resistance", "votes": "4/8", "representative_class": "PUMP_DEGRADATION", "use_in_notebook": "H1 메인 신호"},
    {"feature": "filter_delta_p_kpa", "votes": "2/8", "representative_class": "TIP_CLOG", "use_in_notebook": "H1 보조축"},
    {"feature": "zone1_flow_l_min", "votes": "3/8", "representative_class": "FLOW_DROP", "use_in_notebook": "H1/H2 보조 해석"},
    {"feature": "drain_ec_ds_m", "votes": "2/8", "representative_class": "RPM_INSTABILITY", "use_in_notebook": "H2 메인 반응 신호"},
    {"feature": "zone1_substrate_ec_ds_m", "votes": "2/8", "representative_class": "FLOW_DROP", "use_in_notebook": "H2 보조 zone 신호"},
    {"feature": "zone1_substrate_moisture_pct", "votes": "2/8", "representative_class": "FLOW_DROP", "use_in_notebook": "H2 보조 zone 신호"},
    {"feature": "motor_current_a", "votes": "2/8", "representative_class": "FILTER_CLOG", "use_in_notebook": "H3 보조 지표"},
    {"feature": "bearing_vibration_rms_mm_s", "votes": "3/8", "representative_class": "FILTER_CLOG", "use_in_notebook": "H3 보조 지표"},
    {"feature": "motor_temperature_c", "votes": "4/8", "representative_class": "TIP_CLOG", "use_in_notebook": "H3 보조 지표"},
]

HYPOTHESIS_FEATURE_MAP = [
    {"가설": "H1 관로/팁 막힘", "통계 기법": "Granger", "메인 변수": "zone1_resistance -> zone1_moisture_response_pct", "보조/Robust 피처": "filter_delta_p_kpa, flow_drop_rate, zone1_flow_l_min", "SHAP 근거": "zone1_resistance 4/8 Robust"},
    {"가설": "H2 채널링/미세 누수", "통계 기법": "Event-window CCF", "메인 변수": "mix_flow_l_min -> abs(Δdrain_ec_ds_m)", "보조/Robust 피처": "zone1_substrate_ec_ds_m, zone1_substrate_moisture_pct", "SHAP 근거": "drain_ec_ds_m 2/8 Robust"},
    {"가설": "H3 펌프 노후화/기계 열화", "통계 기법": "Welch T-test + Mann-Whitney", "메인 변수": "wire_to_water_efficiency", "보조/Robust 피처": "motor_current_a, bearing_vibration_rms_mm_s, motor_temperature_c", "SHAP 근거": "motor_temperature_c 4/8 Robust"},
]

def build_shap_anchor_tables():
    return pd.DataFrame(HYPOTHESIS_FEATURE_MAP), pd.DataFrame(ROBUST_FEATURE_TABLE)


def resolve_optional_path(paths):
    for path in paths:
        if path.exists():
            return path
    return None


def format_p_value(value):
    if pd.isna(value):
        return "NA"
    value = float(value)
    if value <= 0:
        return "< 1e-300"
    if value < 1e-4:
        return f"{value:.2e}"
    if value < 0.01:
        return f"{value:.4f}"
    return f"{value:.3f}"


def build_event_lag_table_with_hits(df_raw, abs_diff_signal, threshold, consecutive_hits):
    on_mask = get_on_mask(df_raw)
    starts = find_isolated_event_starts(on_mask)
    rows = []
    for event_idx, ts in enumerate(starts, start=1):
        onset_ts = first_consecutive_hit(
            signal=abs_diff_signal,
            start_ts=ts,
            threshold=threshold,
            max_lag_minutes=MAX_RESPONSE_LAG_MINUTES,
            consecutive_hits=consecutive_hits,
        )
        observed = onset_ts is not None
        lag_minutes = (onset_ts - ts).total_seconds() / 60 if observed else MAX_RESPONSE_LAG_MINUTES
        rows.append(
            {
                "event_index": event_idx,
                "event_start": ts,
                "lag_minutes": float(lag_minutes),
                "event_observed": int(observed),
            }
        )
    return pd.DataFrame(rows)


def get_ccf_peak(curve):
    valid = curve.dropna(subset=["mean_corr"])
    if valid.empty:
        return {"lag_minutes": None, "mean_corr": None}
    peak_row = valid.loc[valid["mean_corr"].abs().idxmax()]
    return {"lag_minutes": int(peak_row["lag_minutes"]), "mean_corr": float(peak_row["mean_corr"])}


def run_h2_shap_guided(df_raw, phase0_result):
    on_mask = get_on_mask(df_raw)
    isolated_starts = phase0_result["isolated_starts"]
    drain_abs_diff = phase0_result["drain_abs_diff"]
    threshold = phase0_result["threshold_info"]["threshold_mad"]

    one_hit_lag_table = build_event_lag_table_with_hits(df_raw, drain_abs_diff, threshold, consecutive_hits=1)
    two_hit_lag_table = build_event_lag_table_with_hits(df_raw, drain_abs_diff, threshold, consecutive_hits=2)
    one_hit_failure_rate = float(1 - one_hit_lag_table["event_observed"].mean())
    two_hit_failure_rate = float(1 - two_hit_lag_table["event_observed"].mean())

    split_point = int(np.ceil(len(isolated_starts) / 2))
    early_starts = isolated_starts[:split_point]
    late_starts = isolated_starts[split_point:]
    ccf_curve, ccf_peak = auxiliary_event_ccf(df_raw, isolated_starts, drain_abs_diff, max_lag_minutes=MAX_RESPONSE_LAG_MINUTES)
    early_curve, early_peak = auxiliary_event_ccf(df_raw, early_starts, drain_abs_diff, max_lag_minutes=MAX_RESPONSE_LAG_MINUTES)
    late_curve, late_peak = auxiliary_event_ccf(df_raw, late_starts, drain_abs_diff, max_lag_minutes=MAX_RESPONSE_LAG_MINUTES)

    observed_lags = two_hit_lag_table.loc[two_hit_lag_table["event_observed"] == 1].copy()
    observed_lags["group"] = np.where(observed_lags["event_index"] <= split_point, "초기 절반", "후기 절반")
    early_obs = observed_lags.loc[observed_lags["group"] == "초기 절반", "lag_minutes"]
    late_obs = observed_lags.loc[observed_lags["group"] == "후기 절반", "lag_minutes"]
    if len(early_obs) > 0 and len(late_obs) > 0:
        lag_mw = mannwhitneyu(early_obs, late_obs, alternative="two-sided")
        lag_mw_p = float(lag_mw.pvalue)
    else:
        lag_mw_p = np.nan

    start_time_mode = isolated_starts.to_series().dt.strftime("%H:%M").mode().iloc[0]
    starts_per_day_mode = pd.Series(isolated_starts.date).value_counts().mode().iloc[0]

    summary = pd.DataFrame(
        [
            {"점검/통계": "이벤트 구조", "값": f"{len(isolated_starts)}개, 하루 {starts_per_day_mode}회, 시작시각 mode {start_time_mode}", "비판적 리뷰": "event index와 운영일이 같은 축"},
            {"점검/통계": "onset 규칙 민감도", "값": f"1-hit 실패율 {one_hit_failure_rate:.3f} / 2-hit 실패율 {two_hit_failure_rate:.3f}", "비판적 리뷰": "같은 데이터라도 규칙을 엄격하게 하면 실패율이 크게 바뀜"},
            {"점검/통계": "전체 CCF peak", "값": f"{ccf_peak['lag_minutes']}분 / corr={ccf_peak['mean_corr']:.3f}" if ccf_peak["lag_minutes"] is not None else "NA", "비판적 리뷰": "corr 절대값이 작으면 강한 증거가 아님"},
            {"점검/통계": "초기 vs 후기 CCF peak", "값": f"{early_peak['lag_minutes']}분 -> {late_peak['lag_minutes']}분" if early_peak["lag_minutes"] is not None and late_peak["lag_minutes"] is not None else "NA", "비판적 리뷰": f"관측 lag 비교 MW p={format_p_value(lag_mw_p)}"},
            {"점검/통계": "최종 판정", "값": "탐색적 분석 유지" if two_hit_failure_rate > 0.30 else "제한적 해석 가능", "비판적 리뷰": "2-hit 실패율이 30%를 넘으면 confirmatory claim 금지"},
        ]
    )

    plt.figure(figsize=(9, 4))
    sns.lineplot(data=ccf_curve.assign(group="전체"), x="lag_minutes", y="mean_corr", label="전체", marker="o")
    sns.lineplot(data=early_curve.assign(group="초기"), x="lag_minutes", y="mean_corr", label="초기 절반", marker="o")
    sns.lineplot(data=late_curve.assign(group="후기"), x="lag_minutes", y="mean_corr", label="후기 절반", marker="o")
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.title("H2 Event-window CCF: mix_flow vs |Δdrain_ec|")
    plt.xlabel("lag (minutes)")
    plt.ylabel("mean correlation")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "h2_event_window_ccf.png", dpi=150, bbox_inches="tight")
    plt.close()

    note_md = "\n".join(
        [
            "### H2 발표 포인트",
            f"- `drain_ec_ds_m` hold 중앙값은 `{exact_hold_run_quantiles(df_raw['drain_ec_ds_m']).loc[0.5]:.1f}분`이고 이벤트 수는 `{len(isolated_starts)}`개입니다.",
            f"- 하지만 onset 실패율이 `1-hit {one_hit_failure_rate:.3f}`에서 `2-hit {two_hit_failure_rate:.3f}`로 크게 뛰어 규칙 민감도가 큽니다.",
            f"- 전체 CCF peak는 `{ccf_peak['lag_minutes']}분 / corr={ccf_peak['mean_corr']:.3f}` 수준이라 강한 정합성 신호라고 보기 어렵습니다.",
            "- 따라서 H2는 채널링/미세 누수의 확증적 검정이 아니라 탐색적 패턴 분석으로만 유지합니다.",
        ]
    )

    return {
        "summary": summary,
        "zone_support": phase0_result["zone_summary"],
        "note_md": note_md,
        "one_hit_failure_rate": one_hit_failure_rate,
        "two_hit_failure_rate": two_hit_failure_rate,
        "ccf_peak": ccf_peak,
        "lag_mw_p": lag_mw_p,
    }


def write_shap_guided_summary(csv_path, phase0_result, h1_result, h2_result, h3_result):
    lines = [
        "# SHAP-Guided Hypothesis Testing Summary",
        "",
        f"- 입력 파일: `{csv_path}`",
        f"- 유의수준: `{ALPHA}`",
        "",
        "## SHAP 앵커",
        dataframe_to_simple_markdown(pd.DataFrame(HYPOTHESIS_FEATURE_MAP)),
        "",
        "## H1",
        dataframe_to_simple_markdown(h1_result["summary"]),
        "",
        "## H2",
        dataframe_to_simple_markdown(h2_result["summary"]),
        "",
        "## H3",
        dataframe_to_simple_markdown(h3_result["group_present"]),
        "",
        dataframe_to_simple_markdown(h3_result["test_present"]),
        "",
        "## 제한해서 말할 점",
        "- H1은 파생변수 구조가 있어 유의성을 곧바로 물리 인과로 번역하면 안 됩니다.",
        f"- H2는 2-hit onset 실패율이 `{h2_result['two_hit_failure_rate']:.3f}`라 탐색적 분석으로만 유지합니다.",
        "- H3는 세척 전후 실험이 아니라 초기 7일 vs 후기 7일 proxy 비교입니다.",
    ]
    (OUT_DIR / "shap_guided_hypothesis_summary.md").write_text("\n".join(lines), encoding="utf-8-sig")


def run_h3_shap_guided(df_feat):
    active = df_feat.loc[df_feat["pump_on_proxy"] == 1].copy()
    active["date"] = active.index.date
    daily_mean = active.groupby("date")[["wire_to_water_efficiency", "motor_current_a", "bearing_vibration_rms_mm_s", "motor_temperature_c"]].mean()
    daily_mean["day_idx"] = np.arange(len(daily_mean))

    group_a = daily_mean.head(WEEK_DAYS).copy()
    group_b = daily_mean.tail(WEEK_DAYS).copy()

    welch = ttest_ind(group_a["wire_to_water_efficiency"], group_b["wire_to_water_efficiency"], equal_var=False)
    mw = mannwhitneyu(group_a["wire_to_water_efficiency"], group_b["wire_to_water_efficiency"], alternative="two-sided")

    trend_present = pd.DataFrame(
        [
            {"지표": "wire_to_water_efficiency", "운영일과의 상관": float(np.corrcoef(daily_mean["day_idx"], daily_mean["wire_to_water_efficiency"])[0, 1])},
            {"지표": "motor_current_a", "운영일과의 상관": float(np.corrcoef(daily_mean["day_idx"], daily_mean["motor_current_a"])[0, 1])},
            {"지표": "bearing_vibration_rms_mm_s", "운영일과의 상관": float(np.corrcoef(daily_mean["day_idx"], daily_mean["bearing_vibration_rms_mm_s"])[0, 1])},
            {"지표": "motor_temperature_c", "운영일과의 상관": float(np.corrcoef(daily_mean["day_idx"], daily_mean["motor_temperature_c"])[0, 1])},
        ]
    )

    group_present = pd.DataFrame(
        [
            {
                "구간": "초기 7일",
                "wire_to_water_efficiency 평균": float(group_a["wire_to_water_efficiency"].mean()),
                "motor_current_a 평균": float(group_a["motor_current_a"].mean()),
                "bearing_vibration_rms_mm_s 평균": float(group_a["bearing_vibration_rms_mm_s"].mean()),
                "motor_temperature_c 평균": float(group_a["motor_temperature_c"].mean()),
            },
            {
                "구간": "후기 7일",
                "wire_to_water_efficiency 평균": float(group_b["wire_to_water_efficiency"].mean()),
                "motor_current_a 평균": float(group_b["motor_current_a"].mean()),
                "bearing_vibration_rms_mm_s 평균": float(group_b["bearing_vibration_rms_mm_s"].mean()),
                "motor_temperature_c 평균": float(group_b["motor_temperature_c"].mean()),
            },
        ]
    )

    test_present = pd.DataFrame(
        [
            {
                "비교 방식": "초기 7일 vs 후기 7일 proxy 비교",
                "Welch p-value": float(welch.pvalue),
                "Mann-Whitney p-value": float(mw.pvalue),
                "비판적 리뷰": "cleaning_event_flag가 없어 세척 전후 실험이 아니라 시간 proxy 비교",
            }
        ]
    )

    plot_df = pd.concat([group_a.assign(group="초기 7일"), group_b.assign(group="후기 7일")]).reset_index(names="date")
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=plot_df, x="group", y="wire_to_water_efficiency")
    plt.title("H3: wire_to_water_efficiency 비교")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "h3_ttest.png", dpi=150, bbox_inches="tight")
    plt.close()

    return {
        "trend_present": trend_present,
        "group_present": group_present,
        "test_present": test_present,
        "test_summary": pd.DataFrame([{"welch_t_p_value": float(welch.pvalue), "mann_whitney_p_value": float(mw.pvalue)}]),
    }


## 1. 데이터 로드

먼저 원시데이터를 읽습니다.


In [ ]:
csv_path = resolve_csv_path(CANDIDATE_PATHS)
df_raw = load_raw_data(csv_path)

print('CSV_PATH:', csv_path)
print('Raw shape:', df_raw.shape)
display(df_raw.head())


## 2. 10분 집계와 파생변수 생성

- **H1**과 **H3**는 10분 집계 데이터를 씁니다.
- **H2**는 변화점 손실을 막기 위해 1분 원시데이터를 그대로 씁니다.


In [ ]:
df_10m = aggregate_10m_tumbling(df_raw)
df_feat = create_features_after_agg(df_10m)

print('10-minute aggregated shape:', df_10m.shape)
print('Feature shape:', df_feat.shape)
display(df_feat[['pump_rpm', 'filter_delta_p_kpa', 'wire_to_water_efficiency', 'zone1_resistance', 'zone1_moisture_response_pct']].head())


## 3. SHAP/Robust 피처 앵커

여기서는 `shap_pipeline.ipynb`에서 이미 뽑아 둔 Robust 피처를 기준으로
이번 발표용 3개 가설이 왜 선택되었는지 먼저 고정합니다.


In [ ]:
hypothesis_feature_df, robust_feature_df = build_shap_anchor_tables()
robust_img_path = resolve_optional_path(ROBUST_IMG_CANDIDATES)
shap_img_path = resolve_optional_path(SHAP_IMG_CANDIDATES)

display(hypothesis_feature_df)
display(robust_feature_df)
if robust_img_path is not None:
    display(Image(filename=str(robust_img_path)))
if shap_img_path is not None:
    display(Image(filename=str(shap_img_path)))


## 4. H2를 하기 전 비판적 점검

H2는 CCF를 바로 해석하면 안 됩니다.
먼저 다음을 확인합니다.

1. `drain_ec_ds_m` 유효 갱신 주기
2. isolated event 수
3. MAD 기반 threshold
4. onset 규칙 민감도
5. zone 보조 신호 가용성


In [ ]:
phase0_result = run_phase0_checks(df_raw)

display(phase0_result['phase0_summary'])
display(phase0_result['zone_summary'])
print('H2 메인 게이트 통과 여부:', phase0_result['h2_main_pass'])


## 5. 검정 1. Granger Causality Test

기존 노트북의 H1 구조를 유지하되,
SHAP에서 반복적으로 등장한 `zone1_resistance`와 `filter_delta_p_kpa`를 중심 축으로 다시 읽습니다.


In [ ]:
h1_result = run_h1(df_feat)
display(h1_result['summary'])

from IPython.display import Image
Image(filename=str(OUT_DIR / 'h1_granger.png'))


## 6. 검정 2. Event-window CCF

사용자가 원한 방법론은 CCF입니다.
다만 이 데이터는 하루 1회 고정 공급 블록 구조라서, 펌프 on/off 스텝 자체를 그대로 CCF에 넣으면 정보량이 낮습니다.

그래서 이번 셀은 다음 순서로 정리합니다.
- 공급 신호 proxy: `mix_flow_l_min`
- 반응 신호: `|Δdrain_ec_ds_m|`
- event-window CCF 계산
- onset 1-hit vs 2-hit 민감도와 함께 결과 해석


In [ ]:
h2_result = run_h2_shap_guided(df_raw, phase0_result)

display(h2_result["summary"])
display(h2_result["zone_support"])
display(Markdown(h2_result["note_md"]))


In [ ]:
Image(filename=str(OUT_DIR / "h2_event_window_ccf.png"))


### H2 중간 해석

여기서는 결론을 세게 말하지 않습니다.

- H2는 CCF를 쓰더라도 event 구조와 onset 규칙에 민감합니다.
- 따라서 채널링/미세 누수의 확증적 증명보다,
  **반복 공급 이벤트에 대한 drain 반응 패턴이 어떻게 보이는가**에 초점을 둡니다.
- 이번 데이터에서는 H2를 **탐색적 분석**으로 유지하는 편이 안전합니다.


## 7. 검정 3. T-test / Mann-Whitney U

기존 H3 구조를 유지하되,
`wire_to_water_efficiency`를 중심으로 `motor_current_a`, `bearing_vibration_rms_mm_s`, `motor_temperature_c`를 함께 읽습니다.


In [ ]:
h3_result = run_h3_shap_guided(df_feat)

display(h3_result['trend_present'])
display(h3_result['group_present'])
display(h3_result['test_present'])
Image(filename=str(OUT_DIR / 'h3_ttest.png'))


## 8. 최종 요약

마지막 셀은 SHAP 근거와 통계 검정을 한 번에 묶어 발표용 요약본으로 출력합니다.


In [ ]:
write_shap_guided_summary(csv_path, phase0_result, h1_result, h2_result, h3_result)
summary_path = OUT_DIR / 'shap_guided_hypothesis_summary.md'
summary_text = summary_path.read_text(encoding='utf-8-sig')
display(Markdown(summary_text))
print('summary saved to:', summary_path)
